# Final Dataset Construction

In [ ]:
import pandas as pd
from tqdm import tqdm
import ast

import requests
import json
from urllib.parse import urlencode
import orjson

import warnings
warnings.filterwarnings("ignore")

In [2]:
BASE_URL = "https://api.openalex.org/works"

selected_fields = ["doi", "publication_year", "language", "indexed_in", "primary_location", "best_oa_location", "open_access", "authorships", 
                   "corresponding_author_ids", "corresponding_institution_ids", "apc_list", "apc_paid", "cited_by_count", "primary_topic", "awards", "funders"]

batch_size = 100

interest_fields = ['doi', 'publication_year', 'language', 'authorships', 'primary_location', 'open_access', 'apc_paid']
keys = ['doi', 'publication_year', 'language', 'issn_l', 'oa_status', 'apc_paid', 'corresponding', 'countries']


def fetch_by_ids(id_list):
    doi_filter = "doi:" + "|".join(id_list)

    filters = [
        doi_filter,
        "indexed_in:crossref",
        "type:article|review",
    ]

    params = {
        "filter": ",".join(filters),
        "select": ",".join(selected_fields),
        "per_page": 200
    }

    url = f"{BASE_URL}?{urlencode(params)}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
        return data.get("results", [])
    except (requests.exceptions.JSONDecodeError, requests.exceptions.RequestException) as e:
        print("Request error or JSON decode error:", e)
        return []

def extract_dois_from_jsonl(file_path):
    records = []
    with open(file_path) as f:
        for line in f:
            rec = orjson.loads(line)
            if not rec.get("doi"):
                continue
            filtered = {k: rec.get(k) for k in interest_fields}

            pl = filtered.get("primary_location") or {}
            source = pl.get("source") or {}
            filtered["issn_l"] = source.get("issn_l")

            authorships = filtered.get("authorships") or []
            filtered["corresponding"] = [auth.get("is_corresponding") for auth in authorships]
            filtered["countries"] = [auth.get("countries") for auth in authorships]
                
            o_a = filtered.get("open_access") or {}
            filtered["oa_status"] = o_a.get("oa_status")

            records.append({k: filtered.get(k) for k in keys})
    return records

def safe_literal_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

## Initial Download Processing

In [3]:
interest = ['doi', 'publication_year', 'language', 'issn_l', 'oa_status', 'apc_paid', 'corresponding', 'countries']

df_oa = []
for year in tqdm(range(2013, 2026)):
    df_oa_tmp = pd.read_csv(f'../data/interim/oa_initial_{year}_v4.csv').drop(columns = 'Unnamed: 0')[interest]
    df_oa_tmp['doi'] = df_oa_tmp['doi'].apply(lambda x: x[16:].lower())
    df_oa.append(df_oa_tmp)
df_oa = pd.concat((df_oa), ignore_index = True)
df_oa

100%|██████████| 13/13 [00:26<00:00,  2.01s/it]


,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries
0,10.1016/j.cell.2013.05.039,2013,en,0092-8674,bronze,NaN,"[False, False, False, True, False]","[['ES'], ['ES'], ['DE', 'GB'], ['ES'], ['FR']]"
1,10.1051/0004-6361/201322068,2013,en,0004-6361,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['DE'], ['US'], ['FR', 'US'], ['FR', 'US'], [..."
2,10.1038/nature12477,2013,en,0028-0836,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['GB'], ['GB'], ['GB'], ['CA', 'GB'], ['GB'],..."
3,10.1097/ccm.0b013e31827e83af,2013,en,0090-3493,closed,NaN,"[True, False, False, False, False, False, Fals...","[['US'], ['US'], ['GB'], ['FR'], ['DE'], ['US'..."
4,10.1083/jcb.201211138,2013,en,0021-9525,bronze,NaN,"[True, False]","[['FR'], ['NL']]"
...,...,...,...,...,...,...,...,...
1805198,10.1109/acoit66109.2025.11436828,2025,NaN,NaN,closed,NaN,"[True, False, False, False, False]","[['IE'], ['US'], ['FR'], [], ['FI']]"
1805199,10.23919/cnc-usnc-ursi64444.2025.11419964,2025,NaN,NaN,closed,NaN,"[True, False]","[['CN'], ['FR']]"
1805200,10.1109/etcm67548.2025.11304322,2025,NaN,NaN,closed,NaN,"[True, False, False, False, False]","[['VE'], ['EC'], ['EC'], ['CN'], ['FR']]"
1805201,10.1093/eurpub/ckaf251,2025,en,1101-1262,gold,"{'value': 2553, 'currency': 'EUR', 'value_usd'...",[True],[['FR']]


## Integrate BSO

To increment the perimeter

In [4]:
interest = ['doi', 'bso_country_corrected', 'genre', 'openalex_id', 'has_fr_corresponding']
records = []
for year in tqdm(range(2013, 2026)):
    with open(f"../data/external/BSO/bso-publications-latest_split_{year}_enriched.jsonl", "rb") as f:
        for line in f:
            rec = orjson.loads(line)
            filtered = {k: rec.get(k) for k in interest} # Keep only the fields we care about
            if not filtered.get("doi"): # Skip records without DOI
                continue
            if 'fr' not in (filtered.get('bso_country_corrected') if isinstance(filtered.get('bso_country_corrected'), list) else [filtered.get('bso_country_corrected')] if filtered.get('bso_country_corrected') else []):
                continue
            if any(g in (filtered['genre'] if isinstance(filtered['genre'], list) else [filtered['genre']]) for g in ['journal-article', 'proceedings']):
                records.append(filtered)
                continue
   
df = pd.DataFrame(records)
df.to_csv('../data/interim/bso.csv', index = False)
df

100%|██████████| 13/13 [04:17<00:00, 19.83s/it]


,doi,bso_country_corrected,genre,openalex_id,has_fr_corresponding
0,10.1016/j.aforl.2013.06.122,"[other, fr]",journal-article,W2064129646,False
1,10.1016/j.rmr.2012.10.076,"[other, fr]",journal-article,W1966675032,False
2,10.4000/cem.13194,"[other, fr]",journal-article,None,True
3,10.1016/j.inoche.2013.04.002,"[fr, other]",journal-article,W2080941060,False
4,10.1016/j.nuclphysa.2013.02.010,"[fr, other]",journal-article,W2062167050,False
...,...,...,...,...,...
1834458,10.1016/j.ces.2025.121516,[fr],journal-article,None,False
1834459,10.1016/j.compeleceng.2025.110714,[fr],journal-article,None,False
1834460,10.1038/s41597-025-04563-2,[fr],journal-article,W4407772959,False
1834461,10.1080/20780389.2025.2568429,[fr],journal-article,None,False


In [4]:
df = pd.read_csv('../data/interim/bso.csv')
df['doi'] = df['doi'].apply(lambda x: x.lower())

df_oa_comb = df_oa.merge(df, on = 'doi', how = 'left')
df_oa_comb['BSO'] = df_oa_comb['openalex_id'].notna()
df_oa_comb = df_oa_comb.drop(columns = ['bso_country_corrected', 'genre', 'openalex_id'])
df_oa_comb

,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,has_fr_corresponding,BSO
0,10.1016/j.cell.2013.05.039,2013,en,0092-8674,bronze,NaN,"[False, False, False, True, False]","[['ES'], ['ES'], ['DE', 'GB'], ['ES'], ['FR']]",False,True
1,10.1051/0004-6361/201322068,2013,en,0004-6361,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['DE'], ['US'], ['FR', 'US'], ['FR', 'US'], [...",False,True
2,10.1038/nature12477,2013,en,0028-0836,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['GB'], ['GB'], ['GB'], ['CA', 'GB'], ['GB'],...",False,True
3,10.1097/ccm.0b013e31827e83af,2013,en,0090-3493,closed,NaN,"[True, False, False, False, False, False, Fals...","[['US'], ['US'], ['GB'], ['FR'], ['DE'], ['US'...",NaN,False
4,10.1083/jcb.201211138,2013,en,0021-9525,bronze,NaN,"[True, False]","[['FR'], ['NL']]",False,True
...,...,...,...,...,...,...,...,...,...,...
1805261,10.1109/acoit66109.2025.11436828,2025,NaN,NaN,closed,NaN,"[True, False, False, False, False]","[['IE'], ['US'], ['FR'], [], ['FI']]",NaN,False
1805262,10.23919/cnc-usnc-ursi64444.2025.11419964,2025,NaN,NaN,closed,NaN,"[True, False]","[['CN'], ['FR']]",NaN,False
1805263,10.1109/etcm67548.2025.11304322,2025,NaN,NaN,closed,NaN,"[True, False, False, False, False]","[['VE'], ['EC'], ['EC'], ['CN'], ['FR']]",NaN,False
1805264,10.1093/eurpub/ckaf251,2025,en,1101-1262,gold,"{'value': 2553, 'currency': 'EUR', 'value_usd'...",[True],[['FR']],NaN,False


### Download the missing

Most of the dois are not articles or reviews

In [5]:
df_bso_oa = df[~df.openalex_id.isna()]
missing = df_bso_oa[~df_bso_oa.doi.str.lower().isin(df_oa_comb.doi)].doi.unique()

all_fetched = []
for i in tqdm(range(0, len(missing), batch_size)):
    batch = missing[i:i+batch_size]
    results = fetch_by_ids(batch)
    all_fetched.extend(results)
    
output_file = "../data/interim/FranceInitialAPI/openalex_missing.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for w in all_fetched:
        f.write(json.dumps(w) + "\n")

print(f"Saved {len(all_fetched)} records to {output_file}")

100%|██████████| 1177/1177 [31:13<00:00,  1.59s/it] 


Saved 18542 records to ../data/interim/FranceInitialAPI/openalex_missing.jsonl


In [5]:
records_tmp = extract_dois_from_jsonl(f"../data/interim/FranceInitialAPI/openalex_missing.jsonl")

df_oa_missing = pd.DataFrame(records_tmp)
df_oa_missing = df_oa_missing[(df_oa_missing['publication_year'] >= 2013) & (df_oa_missing['publication_year'] <= 2025)].reset_index(drop = True)
df_oa_missing['doi'] = df_oa_missing['doi'].apply(lambda x: x[16::].lower())
df_oa_missing = df_oa_missing.merge(df[['doi', 'has_fr_corresponding']], on = 'doi', how = 'left')
print(df_oa_missing.apply(lambda row: True if isinstance(row['countries'], list) and all(len(sub) == 0 for sub in row['countries']) and row['has_fr_corresponding'] == True else False, axis = 1).sum())
df_oa_missing["countries"] = df_oa_missing.apply(lambda row: [['FR'] for _ in row['countries']] if isinstance(row['countries'], list) and all(len(sub) == 0 for sub in row['countries']) and row['has_fr_corresponding'] == True else row['countries'], axis = 1) 
# IN ORDER TO EFFECTIVELY INCORPORATE FROM BSO (I "INVENT" THAT EVERYTHING IS FRENCH) IF has_fr_corresponding == true
df_oa_missing = df_oa_missing.drop(columns = 'has_fr_corresponding')
df_oa_missing['Added'] = 'BSO'
df_oa_missing

1407


,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,Added
0,10.1126/science.1230915,2013,en,0036-8075,closed,None,"[True, False, False, False, False, False, Fals...","[[US], [US], [US], [], [], [US], [US], [US], [...",BSO
1,10.1088/0004-637x/772/1/7,2013,en,0004-637X,bronze,None,"[True, False, False, False]","[[], [], [], []]",BSO
2,10.1038/bjc.2013.764,2013,en,0007-0920,hybrid,"{'value': 3790, 'currency': 'EUR', 'value_usd'...","[True, False, False, False, False, False, Fals...","[[], [], [], [], [CA], [], [], [], [], [CA], [...",BSO
3,10.1080/00207543.2013.849857,2013,en,0020-7543,closed,None,"[False, False, True]","[[RU], [], []]",BSO
4,10.1080/21650349.2013.754657,2013,en,2165-0349,green,None,[True],[[GB]],BSO
...,...,...,...,...,...,...,...,...,...
18480,10.1186/s12967-025-06696-9,2025,en,1479-5876,gold,"{'value': 2390, 'currency': 'GBP', 'value_usd'...","[True, False, False, False, False, False, Fals...","[[], [], [DE], [IT], [], [], [DE, GR], [], [],...",BSO
18481,10.1137/23m1616923,2025,en,0363-0129,closed,None,"[True, False, False]","[[], [], [HK]]",BSO
18482,10.1016/j.repbio.2025.100996,2025,en,1642-431X,closed,None,"[False, False, False, False, True]","[[CA], [CA], [CA], [CA], [CA]]",BSO
18483,10.1016/j.addbeh.2025.108493,2025,en,0306-4603,hybrid,"{'value': 4060, 'currency': 'USD', 'value_usd'...","[True, False, False, False, False, False, Fals...","[[], [], [], [], [], [], [ES], [], [], [], []]",BSO


In [6]:
df_oa_comb_2 = df_oa_missing.merge(df, on = 'doi', how = 'left')
df_oa_comb_2['BSO'] = df_oa_comb_2['openalex_id'].notna()
df_oa_comb_2 = df_oa_comb_2.drop(columns = ['bso_country_corrected', 'genre', 'openalex_id'])

df_tmp = pd.concat((df_oa_comb, df_oa_comb_2)).reset_index(drop = True)
df_tmp

,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,has_fr_corresponding,BSO,Added
0,10.1016/j.cell.2013.05.039,2013,en,0092-8674,bronze,NaN,"[False, False, False, True, False]","[['ES'], ['ES'], ['DE', 'GB'], ['ES'], ['FR']]",False,True,NaN
1,10.1051/0004-6361/201322068,2013,en,0004-6361,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['DE'], ['US'], ['FR', 'US'], ['FR', 'US'], [...",False,True,NaN
2,10.1038/nature12477,2013,en,0028-0836,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['GB'], ['GB'], ['GB'], ['CA', 'GB'], ['GB'],...",False,True,NaN
3,10.1097/ccm.0b013e31827e83af,2013,en,0090-3493,closed,NaN,"[True, False, False, False, False, False, Fals...","[['US'], ['US'], ['GB'], ['FR'], ['DE'], ['US'...",NaN,False,NaN
4,10.1083/jcb.201211138,2013,en,0021-9525,bronze,NaN,"[True, False]","[['FR'], ['NL']]",False,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1823746,10.1186/s12967-025-06696-9,2025,en,1479-5876,gold,"{'value': 2390, 'currency': 'GBP', 'value_usd'...","[True, False, False, False, False, False, Fals...","[[], [], [DE], [IT], [], [], [DE, GR], [], [],...",False,True,BSO
1823747,10.1137/23m1616923,2025,en,0363-0129,closed,None,"[True, False, False]","[[], [], [HK]]",False,True,BSO
1823748,10.1016/j.repbio.2025.100996,2025,en,1642-431X,closed,None,"[False, False, False, False, True]","[[CA], [CA], [CA], [CA], [CA]]",False,True,BSO
1823749,10.1016/j.addbeh.2025.108493,2025,en,0306-4603,hybrid,"{'value': 4060, 'currency': 'USD', 'value_usd'...","[True, False, False, False, False, False, Fals...","[[], [], [], [], [], [], [ES], [], [], [], []]",False,True,BSO


### Check if not in Oa is not in OA

In [4]:
missing = df[df.openalex_id.isna()].doi.unique()
print(len(missing))
all_fetched = []
for i in tqdm(range(0, len(missing), batch_size)):
    batch = missing[i:i+batch_size]
    results = fetch_by_ids(batch)
    all_fetched.extend(results)

output_file = "../data/interim/FranceInitialAPI/not_openalex.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for w in all_fetched:
        f.write(json.dumps(w) + "\n")

print(f"Saved {len(all_fetched)} records to {output_file}")

315357


100%|██████████| 3154/3154 [1:11:49<00:00,  1.37s/it]


Saved 302522 records to ../data/interim/FranceInitialAPI/not_openalex.jsonl


In [7]:
records_tmp = extract_dois_from_jsonl(f"../data/interim/FranceInitialAPI/not_openalex.jsonl")
df_not_oa = pd.DataFrame(records_tmp)
df_not_oa = df_not_oa[(df_not_oa['publication_year'] >= 2013) & (df_not_oa['publication_year'] <= 2025)].reset_index(drop = True)
df_not_oa['doi'] = df_not_oa['doi'].apply(lambda x: x[16::].lower())
df_not_oa = df_not_oa.merge(df[['doi', 'has_fr_corresponding']], on = 'doi', how = 'left')
print(df_not_oa.apply(lambda row: True if isinstance(row['countries'], list) and all(len(sub) == 0 for sub in row['countries']) and row['has_fr_corresponding'] == True else False, axis = 1).sum())
df_not_oa["countries"] = df_not_oa.apply(lambda row: [['FR'] for _ in row['countries']] if isinstance(row['countries'], list) and all(len(sub) == 0 for sub in row['countries']) and row['has_fr_corresponding'] == True else row['countries'], axis = 1) 
# IN ORDER TO EFFECTIVELY INCORPORATE FROM BSO (I "INVENT" THAT EVERYTHING IS FRENCH) IF has_fr_corresponding == true
df_not_oa = df_not_oa.drop(columns = 'has_fr_corresponding')
df_not_oa['Added'] = 'BSO'
df_not_oa

47892


,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,Added
0,10.1056/nejmoa1113697,2013,en,0028-4793,bronze,None,"[True, False, False, False, False, False, Fals...","[[CA, SE], [CA], [GB], [DE], [SE], [NL], [NO],...",BSO
1,10.1016/s0140-6736(12)62191-6,2013,en,0140-6736,closed,None,"[True, False, False, False, False, False, Fals...","[[CH], [GB], [AU], [US], [CH], [CH], [PK], [US...",BSO
2,10.1038/nature11913,2013,en,0028-0836,closed,None,"[True, False, False, False, False, False, False]","[[CH], [CH], [CH], [CH], [CH], [CH], [CH]]",BSO
3,10.3389/fimmu.2013.00297,2013,en,1664-3224,gold,"{'value': 1105, 'currency': 'EUR', 'value_usd'...","[True, False, False]","[[], [DE], [DE]]",BSO
4,10.1162/coli_a_00178,2013,en,0891-2017,bronze,None,[True],[[GB]],BSO
...,...,...,...,...,...,...,...,...,...
302153,10.3390/siuj6050067,2025,en,2563-6499,diamond,None,[True],[[FR]],BSO
302154,10.1007/s11579-025-00406-1,2025,en,1862-9660,closed,None,"[True, False, False]","[[FR], [FR], [FR]]",BSO
302155,10.1080/1475939x.2025.2591424,2025,en,1475-939X,closed,None,"[True, False, False, False]","[[IL], [IN], [IN], [FR]]",BSO
302156,10.1016/j.automatica.2025.112672,2025,en,0005-1098,hybrid,"{'value': 3500, 'currency': 'USD', 'value_usd'...","[True, False, False]","[[FR], [FR], [FR]]",BSO


In [8]:
df_not_oa['BSO'] = True
df_final = pd.concat((df_tmp, df_not_oa)).drop_duplicates('doi').reset_index(drop = True)
df_final

,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,has_fr_corresponding,BSO,Added
0,10.1016/j.cell.2013.05.039,2013,en,0092-8674,bronze,NaN,"[False, False, False, True, False]","[['ES'], ['ES'], ['DE', 'GB'], ['ES'], ['FR']]",False,True,NaN
1,10.1051/0004-6361/201322068,2013,en,0004-6361,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['DE'], ['US'], ['FR', 'US'], ['FR', 'US'], [...",False,True,NaN
2,10.1038/nature12477,2013,en,0028-0836,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['GB'], ['GB'], ['GB'], ['CA', 'GB'], ['GB'],...",False,True,NaN
3,10.1097/ccm.0b013e31827e83af,2013,en,0090-3493,closed,NaN,"[True, False, False, False, False, False, Fals...","[['US'], ['US'], ['GB'], ['FR'], ['DE'], ['US'...",NaN,False,NaN
4,10.1083/jcb.201211138,2013,en,0021-9525,bronze,NaN,"[True, False]","[['FR'], ['NL']]",False,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2050316,10.3917/rdna.880.0044,2025,fr,2105-7508,closed,None,"[True, False]","[[], []]",NaN,True,BSO
2050317,10.1021/acsanm.5c02396,2025,en,2574-0970,hybrid,None,"[False, False, False, False, False, False, Tru...","[[IT], [IT], [IT], [IT], [DE], [], [IT], [IT],...",NaN,True,BSO
2050318,10.15381/rpb.v32i3.30576,2025,es,1561-0837,diamond,None,[True],[[FR]],NaN,True,BSO
2050319,10.3390/siuj6050067,2025,en,2563-6499,diamond,None,[True],[[FR]],NaN,True,BSO


## Integreta NA

In [9]:
interest = ['DOI', 'Has_Discount', 'USD_EUR_Yearly_Avg_Conv_Rate', 'Status_Source', 'Deal_Name', 'Deal_Type', 'Year_Publication', 'Eligible_Deal', 'APC_Accounting_Value']

df_na = pd.read_csv('../data/processed/all_couperin.csv', sep = ',', engine = 'python', on_bad_lines="skip")[interest]
print(len(df_na))
df_na = df_na[(df_na.Year_Publication != 'TRUE') & (df_na.Year_Publication.notna())].reset_index(drop = True)
print(len(df_na))
df_na['Year_Publication'] = df_na['Year_Publication'].astype(int)
df_na['DOI'] = df_na['DOI'].apply(lambda x: x.lower())
df_na = df_na[(df_na.Year_Publication >= 2013) & (df_na.Year_Publication <= 2025)].reset_index(drop = True)
df_na = df_na.drop(columns = 'Year_Publication').rename(columns = {'DOI':'doi'})
df_na['Is_Couperin'] = True
print(len(df_na))
df_na

148478
148019
147973


,doi,Has_Discount,USD_EUR_Yearly_Avg_Conv_Rate,Status_Source,Deal_Name,Deal_Type,Eligible_Deal,APC_Accounting_Value,Is_Couperin
0,10.1186/s12859-022-05048-4,NaN,0.9496,NaN,NaN,NaN,NaN,2190.0,True
1,10.1016/j.rmed.2024.107606,NaN,0.9228,Accepted,Elsevier II,Read & Publish,True,0.0,True
2,10.1186/s12890-019-0851-5,NaN,0.8932,NaN,NaN,NaN,NaN,2190.0,True
3,10.1186/s12931-020-01572-0,NaN,0.8756,NaN,NaN,NaN,NaN,2490.0,True
4,10.1007/s11238-018-9668-6,NaN,0.8470,NaN,NaN,NaN,NaN,NaN,True
...,...,...,...,...,...,...,...,...,...
147968,10.5194/tc-17-5241-2023,NaN,0.9280,NaN,NaN,NaN,NaN,1078.0,True
147969,10.5194/wes-8-1319-2023,NaN,0.9280,NaN,NaN,NaN,NaN,1449.0,True
147970,10.5194/wes-8-141-2023,NaN,0.9280,NaN,NaN,NaN,NaN,483.0,True
147971,10.5194/wes-8-1711-2023,NaN,0.9280,NaN,NaN,NaN,NaN,828.0,True


**Download the missing**

In [10]:
missing = (df_na[~df_na.doi.isin(df_final.doi)].doi.dropna().str.strip().str.lower().unique())
print(len(missing))
all_fetched = []
for i in tqdm(range(0, len(missing), batch_size)):
    batch = missing[i:i+batch_size]
    results = fetch_by_ids(batch)
    all_fetched.extend(results)
    
output_file = "../data/interim/FranceInitialAPI/openalex_na_missing.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for w in all_fetched:
        f.write(json.dumps(w) + "\n")

print(f"Saved {len(all_fetched)} records to {output_file}")

6220


100%|██████████| 63/63 [01:24<00:00,  1.34s/it]

Saved 2437 records to ../data/interim/FranceInitialAPI/openalex_na_missing.jsonl


In [10]:
records_tmp = extract_dois_from_jsonl(f"../data/interim/FranceInitialAPI/openalex_na_missing.jsonl")
df_na_not_oa = pd.DataFrame(records_tmp)
df_na_not_oa = df_na_not_oa[(df_na_not_oa['publication_year'] >= 2013) & (df_na_not_oa['publication_year'] <= 2025)].reset_index(drop = True)
df_na_not_oa["countries"] = df_na_not_oa["countries"].apply(lambda lst: [['FR'] for _ in lst] if isinstance(lst, list) and all(len(sub) == 0 for sub in lst) else lst) # IN ORDER TO EFFECTIVELY INCORPORATE FROM BSO (I INVENT THAT EVERYTHING IS FRENCH)
df_na_not_oa['doi'] = df_na_not_oa['doi'].apply(lambda x: x[16::].lower())
df_na_not_oa['Added'] = 'Couperin'
df_na_not_oa

,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,Added
0,10.1080/22221751.2020.1742585,2020,en,2222-1751,gold,"{'value': 2155, 'currency': 'GBP', 'value_usd'...","[False, False, False, False, True, True]","[[CN], [CN], [CN], [CN], [CN], [CN]]",Couperin
1,10.1080/02678292.2020.1825842,2020,en,0267-8292,bronze,None,"[True, False, False, False]","[[DE], [DE], [DE], [DE]]",Couperin
2,10.1016/s0140-6736(25)01682-4,2025,en,0140-6736,hybrid,"{'value': 6830, 'currency': 'USD', 'value_usd'...","[True, False, False, False, False, False, Fals...","[[], [], [], [US], [US], [US], [US], [US], [CH...",Couperin
3,10.1016/j.archger.2024.105737,2024,en,0167-4943,hybrid,"{'value': 3210, 'currency': 'USD', 'value_usd'...","[True, False, False, False, False, False, Fals...","[[], [IR], [IR], [US], [MA], [JP], [CA], []]",Couperin
4,10.1016/j.inffus.2025.103422,2025,en,1566-2535,hybrid,"{'value': 4650, 'currency': 'USD', 'value_usd'...","[True, False, False, False]","[[FR], [FR], [FR], [FR]]",Couperin
...,...,...,...,...,...,...,...,...,...
2432,10.1016/j.virol.2022.02.006,2022,en,0042-6822,closed,None,"[False, False, False, False, False, False, True]","[[EG], [KR], [EG], [EG], [KR], [RU, ZA], [KR]]",Couperin
2433,10.1016/j.scad.2022.08.012,2022,fr,0183-2980,closed,None,[True],[[FR]],Couperin
2434,10.1016/j.scad.2022.08.015,2022,fr,0183-2980,closed,None,[True],[[FR]],Couperin
2435,10.1016/j.scad.2022.08.014,2022,fr,0183-2980,closed,None,[True],[[FR]],Couperin


In [11]:
df_final = pd.concat((df_final, df_na_not_oa)).drop_duplicates('doi').reset_index(drop = True)

df_final = df_final.merge(df_na, on = 'doi', how = 'left')
df_final.to_csv('../data/processed/initial_dataset.csv', index = False)
df_final

,doi,publication_year,language,issn_l,oa_status,apc_paid,corresponding,countries,has_fr_corresponding,BSO,Added,Has_Discount,USD_EUR_Yearly_Avg_Conv_Rate,Status_Source,Deal_Name,Deal_Type,Eligible_Deal,APC_Accounting_Value,Is_Couperin
0,10.1016/j.cell.2013.05.039,2013,en,0092-8674,bronze,NaN,"[False, False, False, True, False]","[['ES'], ['ES'], ['DE', 'GB'], ['ES'], ['FR']]",False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10.1051/0004-6361/201322068,2013,en,0004-6361,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['DE'], ['US'], ['FR', 'US'], ['FR', 'US'], [...",False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10.1038/nature12477,2013,en,0028-0836,bronze,NaN,"[True, False, False, False, False, False, Fals...","[['GB'], ['GB'], ['GB'], ['CA', 'GB'], ['GB'],...",False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10.1097/ccm.0b013e31827e83af,2013,en,0090-3493,closed,NaN,"[True, False, False, False, False, False, Fals...","[['US'], ['US'], ['GB'], ['FR'], ['DE'], ['US'...",NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10.1083/jcb.201211138,2013,en,0021-9525,bronze,NaN,"[True, False]","[['FR'], ['NL']]",False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2052256,10.1016/j.virol.2022.02.006,2022,en,0042-6822,closed,None,"[False, False, False, False, False, False, True]","[[EG], [KR], [EG], [EG], [KR], [RU, ZA], [KR]]",NaN,NaN,Couperin,NaN,0.9496,NaN,Elsevier I,Read + APC remisés,NaN,NaN,True
2052257,10.1016/j.scad.2022.08.012,2022,fr,0183-2980,closed,None,[True],[[FR]],NaN,NaN,Couperin,NaN,0.9496,NaN,NaN,NaN,NaN,NaN,True
2052258,10.1016/j.scad.2022.08.015,2022,fr,0183-2980,closed,None,[True],[[FR]],NaN,NaN,Couperin,NaN,0.9496,NaN,NaN,NaN,NaN,NaN,True
2052259,10.1016/j.scad.2022.08.014,2022,fr,0183-2980,closed,None,[True],[[FR]],NaN,NaN,Couperin,NaN,0.9496,NaN,NaN,NaN,NaN,NaN,True
